In [5]:
# ============================================
# Real OHLCV test for QuantileRiskAgent (FIXED)
# ============================================

import pandas as pd
import yfinance as yf
from dataclasses import asdict

from quantile_risk_agent import (
    QuantileRiskAgent,
    RegimeConfig,
    QRConfig
)

# --------------------------------------------------
# 1) Download REAL OHLCV data
# --------------------------------------------------
ticker = "AAPL"
timeframe = "1h"
period = "6mo"

df = yf.download(
    tickers=ticker,
    interval=timeframe,
    period=period,
    auto_adjust=False,
    progress=False
)

df = df[["Open", "High", "Low", "Close", "Volume"]]
df.dropna(inplace=True)

print("OHLCV shape:", df.shape)
print(df.head())

# --------------------------------------------------
# 2) Configure agent
# --------------------------------------------------
regime_cfg = RegimeConfig(
    quantiles=(0.25, 0.75),
    atr_window=14,
    trend_window=20
)

qr_cfg = QRConfig(
    quantile=0.95,
    horizon_bars=3,
    alpha=0.01
)

agent = QuantileRiskAgent(
    regime_config=regime_cfg,
    qr_config=qr_cfg
)

# --------------------------------------------------
# 3) Run agent
# --------------------------------------------------
output = agent.run(
    df_ohlcv=df,
    ticker=ticker,
    timeframe=timeframe,
    horizon_bars=3
)

# --------------------------------------------------
# 4) Inspect outputs (CORRECT ATTRIBUTES)
# --------------------------------------------------
print("\n=== Quantile Risk Agent Output ===")

print(f"Timestamp           : {output.timestamp}")
print(f"ATR(t)              : {output.atr_t:.6f}")
print(f"ATR z-score         : {output.atr_z:.4f}")

print(f"Volatility level    : {output.level}")
print(f"Volatility trend    : {output.trend}")
print(f"Regime key          : {output.regime_key}")

print(f"Regime factor       : {output.regime_factor:.4f}")
print(f"Quantile factor     : {output.q_factor:.4f}")

print("\nQuantile forecasts:")
for q, v in output.q_forecasts.items():
    try:
        v_float = float(v)
        print(f"  q={q} → {v_float:.6f}")
    except (ValueError, TypeError):
        print(f"  q={q} → {v}")


print("\nMetadata:")
for k, v in output.meta.items():
    print(f"  - {k}: {v}")

# --------------------------------------------------
# 5) Full debug dump (optional)
# --------------------------------------------------
print("\n=== FULL OUTPUT (DEBUG) ===")
for k, v in asdict(output).items():
    print(f"{k}: {v}")


OHLCV shape: (888, 5)
Price                            Open        High         Low       Close  \
Ticker                           AAPL        AAPL        AAPL        AAPL   
Datetime                                                                    
2025-06-20 13:30:00+00:00  198.235001  200.940002  197.529999  198.679993   
2025-06-20 14:30:00+00:00  198.690002  198.690002  196.859604  197.669998   
2025-06-20 15:30:00+00:00  197.710007  198.860001  197.699997  198.490005   
2025-06-20 16:30:00+00:00  198.490005  199.369995  198.270004  198.320007   
2025-06-20 17:30:00+00:00  198.320007  200.190002  198.070007  199.809998   

Price                        Volume  
Ticker                         AAPL  
Datetime                             
2025-06-20 13:30:00+00:00  26036367  
2025-06-20 14:30:00+00:00   6560590  
2025-06-20 15:30:00+00:00   4287672  
2025-06-20 16:30:00+00:00   4286863  
2025-06-20 17:30:00+00:00   5317209  

=== Quantile Risk Agent Output ===
Timestamp           :

In [7]:
# ============================================================
# Extended validation tests for QuantileRiskAgent
# ============================================================

import pandas as pd
import yfinance as yf
from dataclasses import asdict

from quantile_risk_agent import (
    QuantileRiskAgent,
    RegimeConfig,
    QRConfig
)

# --------------------------------------------------
# Test configurations
# --------------------------------------------------
tests = [
    # (ticker, timeframe, quantile)
    ("AAPL", "1h", 0.95),
    ("AAPL", "15m", 0.99),
    ("BTC-USD", "1h", 0.95),
    ("BTC-USD", "4h", 0.99),
    ("EURUSD=X", "1h", 0.95),
]

period = "6mo"

# --------------------------------------------------
# Loop over tests
# --------------------------------------------------
for ticker, timeframe, q in tests:
    print("\n" + "=" * 60)
    print(f"TEST → ticker={ticker}, timeframe={timeframe}, quantile={q}")
    print("=" * 60)

    # 1) Download data
    df = yf.download(
        tickers=ticker,
        interval=timeframe,
        period=period,
        auto_adjust=False,
        progress=False
    )

    df = df[["Open", "High", "Low", "Close", "Volume"]]
    df.dropna(inplace=True)

    if len(df) < 200:
        print("⚠️ Not enough data, skipping.")
        continue

    # 2) Configure agent
    regime_cfg = RegimeConfig(
        quantiles=(0.25, 0.75),
        atr_window=14,
        trend_window=20
    )

    qr_cfg = QRConfig(
        quantile=q,
        horizon_bars=3,
        alpha=0.01
    )

    agent = QuantileRiskAgent(
        regime_config=regime_cfg,
        qr_config=qr_cfg
    )

    # 3) Run agent
    output = agent.run(
        df_ohlcv=df,
        ticker=ticker,
        timeframe=timeframe,
        horizon_bars=3
    )

    # 4) Display key diagnostics
    print(f"ATR z-score        : {output.atr_z:.3f}")
    print(f"Volatility level  : {output.level}")
    print(f"Volatility trend  : {output.trend}")
    print(f"Regime factor     : {output.regime_factor:.3f}")
    print(f"Quantile factor   : {output.q_factor:.3f}")

    print("Quantile forecasts:")
    for qq, val in output.q_forecasts.items():
        try:
            print(f"  q={qq} → {float(val):.4f}")
        except Exception:
            print(f"  q={qq} → {val}")

    # 5) Sanity checks
    if "0.95" in output.q_forecasts and "0.99" in output.q_forecasts:
        assert float(output.q_forecasts["0.99"]) >= float(output.q_forecasts["0.95"]), \
            "❌ Quantile monotonicity violated!"

    if output.trend == "down":
        assert output.regime_factor <= 1.0, \
            "❌ Regime factor should decrease when volatility trend is down!"

    print("✅ Test passed")



TEST → ticker=AAPL, timeframe=1h, quantile=0.95
ATR z-score        : 1.649
Volatility level  : mid
Volatility trend  : down
Regime factor     : 0.855
Quantile factor   : 0.828
Quantile forecasts:
  q=0.5 → 1.7675
  q=0.95 → 2.1394
✅ Test passed

TEST → ticker=AAPL, timeframe=15m, quantile=0.99



1 Failed download:
['AAPL']: YFPricesMissingError('possibly delisted; no price data found  (period=6mo) (Yahoo error = "15m data not available for startTime=1750326177 and endTime=1766396577. The requested range must be within the last 60 days.")')


⚠️ Not enough data, skipping.

TEST → ticker=BTC-USD, timeframe=1h, quantile=0.95
ATR z-score        : 1.305
Volatility level  : mid
Volatility trend  : down
Regime factor     : 0.855
Quantile factor   : 0.510
Quantile forecasts:
  q=0.5 → 499.2203
  q=0.95 → 839.9824
✅ Test passed

TEST → ticker=BTC-USD, timeframe=4h, quantile=0.99
ATR z-score        : -1.185
Volatility level  : low
Volatility trend  : down
Regime factor     : 0.900
Quantile factor   : 0.619
Quantile forecasts:
  q=0.5 → 1107.6538
  q=0.99 → 2177.7229
✅ Test passed

TEST → ticker=EURUSD=X, timeframe=1h, quantile=0.95
ATR z-score        : -0.918
Volatility level  : low
Volatility trend  : down
Regime factor     : 0.900
Quantile factor   : 0.857
Quantile forecasts:
  q=0.5 → 0.0009
  q=0.95 → 0.0012
✅ Test passed


In [8]:
# ============================================================
# Test script: QuantileRiskAgent + RiskManager (real OHLCV)
# ============================================================

import yfinance as yf
import pandas as pd

from quantile_risk_agent import QuantileRiskAgent, RegimeConfig, QRConfig
from risk_manager import RiskManager, RiskManagerConfig, TradeIntent

# --------------------------
# 1) Load real OHLCV
# --------------------------
ticker = "AAPL"
timeframe = "1h"
period = "6mo"

df = yf.download(
    tickers=ticker,
    interval=timeframe,
    period=period,
    auto_adjust=False,
    progress=False,
)

df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()

print("Data shape:", df.shape)
print(df.tail(3))

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)


# --------------------------
# 2) Run QuantileRiskAgent
# --------------------------
regime_cfg = RegimeConfig(
    quantiles=(0.25, 0.75),
    atr_window=14,
    trend_window=20
)

qr_cfg = QRConfig(
    quantile=0.95,
    horizon_bars=3,
    alpha=0.01
)

qra = QuantileRiskAgent(regime_config=regime_cfg, qr_config=qr_cfg)

risk_out = qra.run(
    df_ohlcv=df,
    ticker=ticker,
    timeframe=timeframe,
    horizon_bars=3
)

print("\n=== QuantileRiskOutput (key) ===")
print("level:", risk_out.level, "trend:", risk_out.trend)
print("regime_factor:", risk_out.regime_factor, "q_factor:", risk_out.q_factor)

# --------------------------
# 3) Mock Trading Direction Agent (TDA) intent
#    (Later we'll replace this with real TDA output)
# --------------------------
intent = TradeIntent(
    direction="long",
    baseline_size=1.0,     # e.g., 1 contract/share unit (placeholder)
    confidence=0.62,
    meta={"source": "mock_tda"}
)

# --------------------------
# 4) RiskManager (deterministic live guardrails)
# --------------------------
rm_cfg = RiskManagerConfig(
    lambda_reg=0.5,
    lambda_q=0.5,
    warn_kfinal=1.10,
    veto_kfinal=1.35,
    min_confidence=0.15,
    size_curve=2.0,
    sl_curve=1.0,
    tp_curve=1.0,
    tp_reduce_in_high_risk=True
)

rm = RiskManager(config=rm_cfg)

rm_out = rm.decide(risk_output=risk_out, intent=intent)

print("\n=== RiskManagerOutput ===")
print("k_final:", rm_out.k_final)
print("action:", rm_out.action)
print("decision:", rm_out.explanation["decision"])
print("veto_reasons:", rm_out.explanation["veto_reasons"])

print("\n--- Execution payload ---")
print(rm.to_execution_payload(rm_out))

print("\n--- XAI payload ---")
print(rm.to_xai_payload(rm_out))

# --------------------------
# 5) Minimal sanity assertions
# --------------------------
a = rm_out.action
assert 0.0 <= a.scale_size <= 1.0
assert 0.5 <= a.scale_sl <= 2.0
assert 0.5 <= a.scale_tp <= 2.0
assert a.accept_reject in (0, 1)

# If trend is down, regime_factor often <= 1 (depends on your mapping),
# but at least k_final should be finite.
assert pd.notna(rm_out.k_final) and abs(rm_out.k_final) < 1e6

print("\n✅ RiskManager test passed.")


Data shape: (888, 5)
Price                            Open        High         Low       Close  \
Ticker                           AAPL        AAPL        AAPL        AAPL   
Datetime                                                                    
2025-12-19 18:30:00+00:00  270.760010  271.509888  270.399994  271.369995   
2025-12-19 19:30:00+00:00  271.380005  271.739990  270.174988  270.799988   
2025-12-19 20:30:00+00:00  270.809998  274.576294  269.899994  273.670013   

Price                       Volume  
Ticker                        AAPL  
Datetime                            
2025-12-19 18:30:00+00:00  2544938  
2025-12-19 19:30:00+00:00  3108119  
2025-12-19 20:30:00+00:00  8193309  

=== QuantileRiskOutput (key) ===
level: mid trend: down
regime_factor: 0.855 q_factor: 0.8278403067758188

=== RiskManagerOutput ===
k_final: 0.8414201533879093
action: RiskAction(scale_size=1.0, scale_sl=1.1217736247600067, scale_tp=1.3782263752399933, accept_reject=1)
decision: ACCEPT
veto_

In [10]:
# ============================================================
# Test RiskEnv with real OHLCV + QuantileRiskAgent + RiskManager
# ============================================================

import numpy as np
import pandas as pd
import yfinance as yf

from quantile_risk_agent import QuantileRiskAgent, RegimeConfig, QRConfig
from risk_manager import RiskManager, RiskManagerConfig, TradeIntent
from risk_env import RiskEnv, RiskEnvConfig


# -------------------------
# 1) Download real OHLCV
# -------------------------
ticker = "AAPL"
timeframe = "1h"
period = "6mo"

df = yf.download(
    tickers=ticker,
    interval=timeframe,
    period=period,
    auto_adjust=False,
    progress=False,
)

# Fix MultiIndex columns if needed
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()
print("OHLCV:", df.shape, df.index.min(), "→", df.index.max())


# -------------------------
# 2) Build QuantileRiskAgent + RiskManager
# -------------------------
regime_cfg = RegimeConfig(quantiles=(0.25, 0.75), atr_window=14, trend_window=20)
qr_cfg = QRConfig(quantile=0.95, horizon_bars=3, alpha=0.01)

qra = QuantileRiskAgent(regime_config=regime_cfg, qr_config=qr_cfg)

rm_cfg = RiskManagerConfig(
    lambda_reg=0.5,
    lambda_q=0.5,
    warn_kfinal=1.10,
    veto_kfinal=1.35,
    min_confidence=0.15,
    size_curve=2.0,
    sl_curve=1.0,
    tp_curve=1.0,
    tp_reduce_in_high_risk=True,
)
rm = RiskManager(config=rm_cfg)


# -------------------------
# 3) Providers: risk features + intent
# -------------------------
def risk_feature_provider(t: int, df_slice: pd.DataFrame):
    """
    Compute risk features using only df up to time t (no lookahead).
    For testing, this is OK even if it's slower.
    """
    # need enough rows for agent windows
    if len(df_slice) < 320:
        return {"k_final": 1.0, "regime_factor": 1.0, "q_factor": 1.0, "atr_z": 0.0}

    risk_out = qra.run(df_ohlcv=df_slice, ticker=ticker, timeframe=timeframe, horizon_bars=3)

    # mock TDA intent (for testing only)
    intent = TradeIntent(direction="long", baseline_size=1.0, confidence=0.6)

    rm_out = rm.decide(risk_output=risk_out, intent=intent)

    return {
        "k_final": rm_out.k_final,
        "regime_factor": float(risk_out.regime_factor),
        "q_factor": float(risk_out.q_factor),
        "atr_z": float(risk_out.atr_z),
    }


def intent_provider(t: int, df_slice: pd.DataFrame):
    """
    Simple deterministic intent for testing:
    - long if last return positive, short if negative (toy logic)
    """
    if len(df_slice) < 2:
        return {"direction": "flat", "baseline_size": 0.0, "confidence": 0.0}

    r = float(df_slice["Close"].iloc[-1] / df_slice["Close"].iloc[-2] - 1.0)
    if r > 0:
        return {"direction": "long", "baseline_size": 1.0, "confidence": 0.6}
    elif r < 0:
        return {"direction": "short", "baseline_size": 1.0, "confidence": 0.6}
    return {"direction": "flat", "baseline_size": 0.0, "confidence": 0.0}


# -------------------------
# 4) Create RiskEnv
# -------------------------
env_cfg = RiskEnvConfig(
    initial_capital=10_000.0,
    max_position_units=1.0,
    base_sl_atr=1.5,
    base_tp_atr=2.0,
)

env = RiskEnv(
    df_ohlcv=df,
    cfg=env_cfg,
    risk_feature_provider=risk_feature_provider,
    intent_provider=intent_provider,
)

obs = env.reset(start_index=350)
print("Obs dim:", obs.shape)


# -------------------------
# 5) Run a short roll-out with random actions
# -------------------------
np.random.seed(42)

for step in range(30):
    # Random bounded action:
    # scale_size in [0,1], sl,tp in [0.5,2], accept in [0,1]
    a = np.array([
        np.random.rand(),
        0.5 + 1.5 * np.random.rand(),
        0.5 + 1.5 * np.random.rand(),
        np.random.rand(),
    ], dtype=float)

    obs, reward, done, info = env.step(a)

    print(
        f"step={step:02d} t={info['t']} event={info['event']:<12} "
        f"reward={reward:+.4f} pnl={info['pnl']:+.2f} cap={info['capital']:.2f} "
        f"dd={info['drawdown']:.4f} k_final={info['k_final']:.3f}"
    )

    # Basic sanity checks
    assert np.isfinite(reward)
    assert np.isfinite(info["capital"])
    assert 0.0 <= info["drawdown"] <= 1.0 + 1e-6

    if done:
        print("Episode ended early:", info.get("event"))
        break

print("\n✅ RiskEnv test completed.")


OHLCV: (888, 5) 2025-06-20 13:30:00+00:00 → 2025-12-19 20:30:00+00:00
Obs dim: (47,)
step=00 t=351 event=mark_to_market reward=-0.0630 pnl=-0.25 cap=9999.75 dd=0.0000 k_final=0.825
step=01 t=352 event=mark_to_market reward=-0.0799 pnl=+0.32 cap=10000.07 dd=0.0000 k_final=0.825
step=02 t=353 event=take_profit  reward=-0.0588 pnl=+0.83 cap=10000.90 dd=0.0000 k_final=0.819


d:\these_achraf\xdta\src\xdta\agents\risk_agent\risk_env.py:254: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.atr_z = _zscore(self.atr, window=252)


step=03 t=354 event=hold         reward=-0.0200 pnl=+0.00 cap=10000.90 dd=0.0000 k_final=0.792
step=04 t=355 event=hold         reward=-0.0148 pnl=+0.00 cap=10000.90 dd=0.0000 k_final=0.753
step=05 t=356 event=hold         reward=-0.0117 pnl=+0.00 cap=10000.90 dd=0.0000 k_final=0.744
step=06 t=357 event=mark_to_market reward=-0.0598 pnl=-0.29 cap=10000.61 dd=0.0000 k_final=0.764
step=07 t=358 event=hold         reward=-0.0676 pnl=+0.28 cap=10000.89 dd=0.0000 k_final=0.761
step=08 t=359 event=mark_to_market reward=-0.0371 pnl=+0.08 cap=10000.96 dd=0.0000 k_final=0.766
step=09 t=360 event=hold         reward=-0.0296 pnl=-0.08 cap=10000.88 dd=0.0000 k_final=0.769
step=10 t=361 event=mark_to_market reward=-0.0344 pnl=+0.02 cap=10000.90 dd=0.0000 k_final=0.768
step=11 t=362 event=mark_to_market reward=-0.0256 pnl=+0.01 cap=10000.92 dd=0.0000 k_final=0.764
step=12 t=363 event=mark_to_market reward=-0.0514 pnl=-0.58 cap=10000.34 dd=0.0001 k_final=0.767
step=13 t=364 event=mark_to_market rewar

In [ ]:
# ============================================================
#   TEST: risk_policy.py with RiskEnv (PPO quick training)
# ============================================================

import yfinance as yf
import pandas as pd

from risk_manager import TradeIntent


from quantile_risk_agent import QuantileRiskAgent, RegimeConfig, QRConfig
from risk_manager import RiskManager, RiskManagerConfig
from risk_env import RiskEnv, RiskEnvConfig

from risk_policy import RiskPolicy, RiskPolicyConfig, random_rollout, RiskEnvGymWrapper

import os
os.makedirs("models", exist_ok=True)



# -------------------------
# 1) Download real OHLCV
# -------------------------
ticker = "AAPL"
timeframe = "1h"
period = "6mo"

df = yf.download(
    tickers=ticker,
    interval=timeframe,
    period=period,
    auto_adjust=False,
    progress=False
)

# Fix yfinance MultiIndex columns
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Keep standard OHLCV only
df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()

print("OHLCV:", df.shape, df.index.min(), "→", df.index.max())


# -------------------------
# 2) Build QuantileRiskAgent
# -------------------------
regime_cfg = RegimeConfig(
    quantiles=(0.25, 0.75),
    atr_window=14,
    trend_fraction=0.2,
    trend_window=20  
)

qr_cfg = QRConfig(
    horizon_bars=3,                 
    feature_window=60,
    quantile_levels=(0.5, 0.95),    
    alpha=0.01,
    alpha_mode="vol_scaled",
)

qra = QuantileRiskAgent(regime_config=regime_cfg, qr_config=qr_cfg)


# -------------------------
# 3) Build RiskManager
# -------------------------
rm_cfg = RiskManagerConfig(
    lambda_reg=0.5,
    lambda_q=0.5,
    warn_kfinal=1.10,
    veto_kfinal=1.35,
    min_confidence=0.15,
    size_curve=2.0,
    sl_curve=1.0,
    tp_curve=1.0,
    tp_reduce_in_high_risk=True,
)

rm = RiskManager(rm_cfg)


# -------------------------
# 4) Build RiskEnv
# -------------------------
env_cfg = RiskEnvConfig(
    initial_capital=10_000.0,
    max_position_units=1.0,
    base_sl_atr=1.5,
    base_tp_atr=2.0,
)

def risk_feature_provider(t: int, df: pd.DataFrame):
    out = qra.run(df_ohlcv=df.iloc[: t + 1], horizon_bars=3)
    return {
        "k_final": out.q_factor,
        "regime_factor": out.regime_factor,
        "quantile_factor": out.q_factor,
        "level": out.level,
        "trend": out.trend,
    }

def intent_provider(t: int, df: pd.DataFrame):
    intent = TradeIntent(
        direction="long",
        baseline_size=1.0,
        confidence=0.5,
        meta={"source": "mock_tda"},
    )

    risk_out = qra.run(df_ohlcv=df.iloc[: t + 1], horizon_bars=3)

    rm_out = rm.decide(
        risk_output=risk_out,
        intent=intent,
    )

    return {
        "action": rm_out.action,
        "k_final": rm_out.k_final,
    }


env = RiskEnv(
    df_ohlcv=df,
    cfg=RiskEnvConfig(),
    risk_feature_provider=risk_feature_provider,
    intent_provider=intent_provider,
)


obs = env.reset()
print("Obs dim:", (len(obs) if hasattr(obs, "__len__") else obs))


# -------------------------
# 5) Train PPO or fallback
# -------------------------
policy_cfg = RiskPolicyConfig(
    algo="PPO",               # change to "SAC" later
    total_timesteps=5_000,    # quick sanity train (increase later)
    eval_every_steps=2_500,
    eval_episodes=2,
    verbose=1,
)



try:
    policy = RiskPolicy(policy_cfg)
    model = policy.train(env)
    policy.model.save("models/ppo_risk_20k")


    print("✅ PPO training completed (sanity run).")

except Exception as e:
    print("⚠️ SB3 training unavailable, running random rollout instead.")
    print("Reason:", repr(e))
    random_rollout(env, policy_cfg, n_steps=50, seed=0)


OHLCV: (877, 5) 2025-06-27 13:30:00+00:00 → 2025-12-26 20:30:00+00:00
Obs dim: 71
Using cpu device
-----------------------------
| time/              |      |
|    fps             | 3    |
|    iterations      | 1    |
|    time_elapsed    | 576  |
|    total_timesteps | 2048 |
-----------------------------
[Eval] mean=-0.0300
-----------------------------------------
| time/                   |             |
|    fps                  | 2           |
|    iterations           | 2           |
|    time_elapsed         | 1712        |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.006644347 |
|    clip_fraction        | 0.0933      |
|    clip_range           | 0.2         |
|    entropy_loss         | -5.66       |
|    explained_variance   | -1.61       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.00465    |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.00989    |

In [ ]:
policy = RiskPolicy(policy_cfg)
print(hasattr(policy, "train"))

print(policy.model)

True
None


In [3]:
# ============================================================
#   TEST: risk_policy.py with RiskEnv (PPO quick training)
# ============================================================

import yfinance as yf
import pandas as pd

from risk_manager import TradeIntent


from quantile_risk_agent import QuantileRiskAgent, RegimeConfig, QRConfig
from risk_manager import RiskManager, RiskManagerConfig
from risk_env import RiskEnv, RiskEnvConfig

from risk_policy import RiskPolicy, RiskPolicyConfig, random_rollout, RiskEnvGymWrapper

import os
os.makedirs("models", exist_ok=True)



# -------------------------
# 1) Download real OHLCV
# -------------------------
ticker = "AAPL"
timeframe = "1h"
period = "6mo"

df = yf.download(
    tickers=ticker,
    interval=timeframe,
    period=period,
    auto_adjust=False,
    progress=False
)

# Fix yfinance MultiIndex columns
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Keep standard OHLCV only
df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()

print("OHLCV:", df.shape, df.index.min(), "→", df.index.max())


# -------------------------
# 2) Build QuantileRiskAgent
# -------------------------
regime_cfg = RegimeConfig(
    quantiles=(0.25, 0.75),
    atr_window=14,
    trend_fraction=0.2,
    trend_window=20  
)

qr_cfg = QRConfig(
    horizon_bars=3,                 
    feature_window=60,
    quantile_levels=(0.5, 0.95),    
    alpha=0.01,
    alpha_mode="vol_scaled",
)

qra = QuantileRiskAgent(regime_config=regime_cfg, qr_config=qr_cfg)


# -------------------------
# 3) Build RiskManager
# -------------------------
rm_cfg = RiskManagerConfig(
    lambda_reg=0.5,
    lambda_q=0.5,
    warn_kfinal=1.10,
    veto_kfinal=1.35,
    min_confidence=0.15,
    size_curve=2.0,
    sl_curve=1.0,
    tp_curve=1.0,
    tp_reduce_in_high_risk=True,
)

rm = RiskManager(rm_cfg)


# -------------------------
# 4) Build RiskEnv
# -------------------------
env_cfg = RiskEnvConfig(
    initial_capital=10_000.0,
    max_position_units=1.0,
    base_sl_atr=1.5,
    base_tp_atr=2.0,
)

def risk_feature_provider(t: int, df: pd.DataFrame):
    out = qra.run(df_ohlcv=df.iloc[: t + 1], horizon_bars=3)
    return {
        "k_final": out.q_factor,
        "regime_factor": out.regime_factor,
        "quantile_factor": out.q_factor,
        "level": out.level,
        "trend": out.trend,
    }

def intent_provider(t: int, df: pd.DataFrame):
    intent = TradeIntent(
        direction="long",
        baseline_size=1.0,
        confidence=0.5,
        meta={"source": "mock_tda"},
    )

    risk_out = qra.run(df_ohlcv=df.iloc[: t + 1], horizon_bars=3)

    rm_out = rm.decide(
        risk_output=risk_out,
        intent=intent,
    )

    return {
        "action": rm_out.action,
        "k_final": rm_out.k_final,
    }


env = RiskEnv(
    df_ohlcv=df,
    cfg=RiskEnvConfig(),
    risk_feature_provider=risk_feature_provider,
    intent_provider=intent_provider,
)


obs = env.reset()
print("Obs dim:", (len(obs) if hasattr(obs, "__len__") else obs))


# -------------------------
# 5) Train PPO or fallback
# -------------------------
policy_cfg = RiskPolicyConfig(
    algo="PPO",               
    total_timesteps=20_000,    
    eval_every_steps=4_000,
    eval_episodes=3,
    verbose=1,
)



try:
    policy = RiskPolicy(policy_cfg)
    model = policy.train(env)
    policy.model.save("models/ppo_risk_20k")


    print("✅ PPO training completed (sanity run).")

except Exception as e:
    print("⚠️ SB3 training unavailable, running random rollout instead.")
    print("Reason:", repr(e))
    random_rollout(env, policy_cfg, n_steps=50, seed=0)



# -------------------------
# 6) Train SAC
# -------------------------
policy_cfg = RiskPolicyConfig(
    algo="SAC",
    total_timesteps=20_000,
    eval_every_steps=4_000,
    eval_episodes=3,
    verbose=1,
)

policy = RiskPolicy(policy_cfg)
model = policy.train(env)
policy.model.save("models/sac_risk_20k")

print("✅ SAC training completed (20k).")



OHLCV: (877, 5) 2025-06-30 13:30:00+00:00 → 2025-12-29 20:30:00+00:00
Obs dim: 71
Using cpu device
-----------------------------
| time/              |      |
|    fps             | 2    |
|    iterations      | 1    |
|    time_elapsed    | 789  |
|    total_timesteps | 2048 |
-----------------------------
[Eval] mean=-0.0341
-----------------------------------------
| time/                   |             |
|    fps                  | 1           |
|    iterations           | 2           |
|    time_elapsed         | 2381        |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.007261428 |
|    clip_fraction        | 0.0875      |
|    clip_range           | 0.2         |
|    entropy_loss         | -5.66       |
|    explained_variance   | -1.69       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.0162     |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.00856    |

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3 import SAC

sac_model = SAC.load("models/sac_risk_20k")
ppo_model = PPO.load("models/ppo_risk_20k")

import numpy as np

def evaluate_agent(
    model,
    env,
    n_episodes=3,
    deterministic=True,
):
    import numpy as np

    results = {
        "episode_reward": [],
        "final_capital": [],
        "max_drawdown": [],
        "turnover": [],
    }

    for ep in range(n_episodes):

        # -------- reset (robust) --------
        reset_out = env.reset()
        if isinstance(reset_out, tuple):
            obs = reset_out[0]
        else:
            obs = reset_out

        done = False
        trunc = False

        total_reward = 0.0
        max_dd = 0.0
        total_turnover = 0.0

        while not (done or trunc):
            action, _ = model.predict(obs, deterministic=deterministic)

            step_out = env.step(action)

            # -------- step (robust) --------
            if len(step_out) == 5:
                obs, reward, done, trunc, info = step_out
            else:
                obs, reward, done, info = step_out
                trunc = False

            total_reward += float(reward)
            max_dd = max(max_dd, info.get("drawdown", 0.0))
            total_turnover += info.get("turnover", 0.0)

        results["episode_reward"].append(total_reward)
        results["final_capital"].append(info.get("capital", np.nan))
        results["max_drawdown"].append(max_dd)
        results["turnover"].append(total_turnover)

    return {
        "reward_mean": float(np.mean(results["episode_reward"])),
        "reward_std": float(np.std(results["episode_reward"])),
        "final_capital_mean": float(np.mean(results["final_capital"])),
        "max_drawdown_mean": float(np.mean(results["max_drawdown"])),
        "turnover_mean": float(np.mean(results["turnover"])),
    }



ppo_metrics = evaluate_agent(ppo_model, env, n_episodes=5)
print("PPO:", ppo_metrics)

sac_metrics = evaluate_agent(sac_model, env, n_episodes=5)
print("SAC:", sac_metrics)




NameError: name 'env' is not defined

In [10]:
# ============================================================
#   TEST: risk_policy.py with RiskEnv (PPO quick training)
# ============================================================

import yfinance as yf
import pandas as pd

from risk_manager import TradeIntent


from quantile_risk_agent import QuantileRiskAgent, RegimeConfig, QRConfig
from risk_manager import RiskManager, RiskManagerConfig
from risk_env import RiskEnv, RiskEnvConfig

from risk_policy import RiskPolicy, RiskPolicyConfig, random_rollout, RiskEnvGymWrapper

import os
os.makedirs("models", exist_ok=True)



# -------------------------
# 1) Download real OHLCV
# -------------------------
ticker = "AAPL"
timeframe = "1h"
period = "6mo"

df = yf.download(
    tickers=ticker,
    interval=timeframe,
    period=period,
    auto_adjust=False,
    progress=False
)


In [11]:

# Fix yfinance MultiIndex columns
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Keep standard OHLCV only
df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()

print("OHLCV:", df.shape, df.index.min(), "→", df.index.max())


# -------------------------
# 2) Build QuantileRiskAgent
# -------------------------
regime_cfg = RegimeConfig(
    quantiles=(0.25, 0.75),
    atr_window=14,
    trend_fraction=0.2,
    trend_window=20  
)

qr_cfg = QRConfig(
    horizon_bars=3,                 
    feature_window=60,
    quantile_levels=(0.5, 0.95),    
    alpha=0.01,
    alpha_mode="vol_scaled",
)

qra = QuantileRiskAgent(regime_config=regime_cfg, qr_config=qr_cfg)


# -------------------------
# 3) Build RiskManager
# -------------------------
rm_cfg = RiskManagerConfig(
    lambda_reg=0.5,
    lambda_q=0.5,
    warn_kfinal=1.10,
    veto_kfinal=1.35,
    min_confidence=0.15,
    size_curve=2.0,
    sl_curve=1.0,
    tp_curve=1.0,
    tp_reduce_in_high_risk=True,
)

rm = RiskManager(rm_cfg)


# -------------------------
# 4) Build RiskEnv
# -------------------------
env_cfg = RiskEnvConfig(
    initial_capital=10_000.0,
    max_position_units=1.0,
    base_sl_atr=1.5,
    base_tp_atr=2.0,
)

def risk_feature_provider(t: int, df: pd.DataFrame):
    out = qra.run(df_ohlcv=df.iloc[: t + 1], horizon_bars=3)
    return {
        "k_final": out.q_factor,
        "regime_factor": out.regime_factor,
        "quantile_factor": out.q_factor,
        "level": out.level,
        "trend": out.trend,
    }

def intent_provider(t: int, df: pd.DataFrame):
    # 1) Base trading intent (from signal / TDA)
    intent = TradeIntent(
        direction="long",
        baseline_size=1.0,     # base position size (contracts / units)
        confidence=0.5,
        meta={"source": "mock_tda"},
    )

    # 2) Risk features
    risk_out = qra.run(df_ohlcv=df.iloc[: t + 1], horizon_bars=3)

    # 3) Risk manager decision
    rm_out = rm.decide(
        risk_output=risk_out,
        intent=intent,
    )

    # 4) Apply RM scaling to baseline size
    baseline_size = intent.baseline_size * rm_out.action.scale_size

    # 5) Return EXACTLY what RiskEnv expects
    return {
        # --- REQUIRED ---
        "direction": intent.direction,
        "baseline_size": float(baseline_size),
        "confidence": float(intent.confidence),

        # --- RiskManager controls ---
        "accept_reject": int(rm_out.action.accept_reject),
        "scale_sl": float(rm_out.action.scale_sl),
        "scale_tp": float(rm_out.action.scale_tp),

        # --- Debug / logging ---
        "k_final": float(rm_out.k_final),
        "diagnostics": rm_out.diagnostics,
    }



env = RiskEnv(
    df_ohlcv=df,
    cfg=RiskEnvConfig(),
    risk_feature_provider=risk_feature_provider,
    intent_provider=intent_provider,
)


obs = env.reset()
print("Obs dim:", (len(obs) if hasattr(obs, "__len__") else obs))

OHLCV: (881, 5) 2025-07-07 13:30:00+00:00 → 2026-01-05 20:30:00+00:00
Obs dim: 71


In [12]:
import numpy as np

obs = env.reset()

obs, r, done, info = env.step(np.array([1.0, 1.0, 1.0, 1.0]))

print("reward:", r)
print("turnover:", info.get("turnover"))
print("event:", info.get("event"))
print("capital:", info.get("capital"))

print("debug:", {
    "accept": info.get("accept"),
    "target_units": info.get("target_units"),
    "delta_units": info.get("delta_units"),
    "direction": info.get("direction"),
})


reward: -0.0010089829482257753
turnover: 1.0
event: mark_to_market
capital: 9999.45508525887
debug: {'accept': 1, 'target_units': 1.0, 'delta_units': 1.0, 'direction': 'long'}


In [4]:
obs = env.reset()

for i in range(10):
    obs, r, done, info = env.step(np.array([1.0, 1.0, 1.0, 1.0]))
    print(i, "r=", round(r,6), "pnl=", round(info["pnl"],4), "cap=", round(info["capital"],2))


0 r= -0.00092 pnl= -0.0991 cap= 9999.9
1 r= 6.4e-05 pnl= 0.6445 cap= 10000.55
2 r= 0.000101 pnl= 1.0055 cap= 10001.55
3 r= -8.5e-05 pnl= -0.425 cap= 10001.13
4 r= 5.2e-05 pnl= 0.52 cap= 10001.65
5 r= -0.000207 pnl= -1.035 cap= 10000.61
6 r= 8.5e-05 pnl= 0.855 cap= 10001.47
7 r= -0.000101 pnl= -0.505 cap= 10000.96
8 r= 8e-05 pnl= 0.805 cap= 10001.77
9 r= -5.1e-05 pnl= -0.255 cap= 10001.51


In [6]:
from risk_policy import RiskPolicyConfig, random_rollout

policy_cfg = RiskPolicyConfig(
    algo="PPO",          # peu importe ici
    total_timesteps=1,  # non utilisé par random_rollout
)

env.reset()

random_rollout(
    env,
    policy_cfg,
    n_steps=1000,
    seed=0
)


[random] step=000 r=-0.0019
[random] step=001 r=-0.0037
[random] step=002 r=-0.0023
[random] step=003 r=-0.0018
[random] step=004 r=-0.0014
[random] step=005 r=-0.0030
[random] step=006 r=-0.0019
[random] step=007 r=-0.0024
[random] step=008 r=-0.0009
[random] step=009 r=-0.0012
[random] step=010 r=-0.0014
[random] step=011 r=-0.0029
[random] step=012 r=-0.0026
[random] step=013 r=-0.0026
[random] step=014 r=-0.0017
[random] step=015 r=-0.0025
[random] step=016 r=-0.0010
[random] step=017 r=-0.0016
[random] step=018 r=-0.0025
[random] step=019 r=-0.0009
[random] step=020 r=-0.0030
[random] step=021 r=-0.0013
[random] step=022 r=-0.0007
[random] step=023 r=-0.0004
[random] step=024 r=-0.0006
[random] step=025 r=-0.0021
[random] step=026 r=-0.0013
[random] step=027 r=-0.0020
[random] step=028 r=-0.0035
[random] step=029 r=-0.0019
[random] step=030 r=-0.0025
[random] step=031 r=-0.0016
[random] step=032 r=-0.0017
[random] step=033 r=-0.0025
[random] step=034 r=-0.0011
[random] step=035 r=

-1.9947902853963841

In [1]:
# ============================================================
#   FINAL TEST: RiskEnv + RiskPolicy (PPO & SAC)
# ============================================================

import yfinance as yf
import pandas as pd
import os

from quantile_risk_agent import QuantileRiskAgent, RegimeConfig, QRConfig
from risk_manager import RiskManager, RiskManagerConfig, TradeIntent
from risk_env import RiskEnv, RiskEnvConfig
from risk_policy import RiskPolicy, RiskPolicyConfig

os.makedirs("models", exist_ok=True)

# -------------------------
# 1) Download OHLCV
# -------------------------
df = yf.download(
    tickers="AAPL",
    interval="1h",
    period="6mo",
    auto_adjust=False,
    progress=False,
)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()

# -------------------------
# 2) Quantile Risk Agent
# -------------------------
qra = QuantileRiskAgent(
    regime_config=RegimeConfig(
        quantiles=(0.25, 0.75),
        atr_window=14,
        trend_fraction=0.2,
        trend_window=20,
    ),
    qr_config=QRConfig(
        horizon_bars=3,
        feature_window=60,
        quantile_levels=(0.5, 0.95),
        alpha=0.01,
        alpha_mode="vol_scaled",
    ),
)

# -------------------------
# 3) Risk Manager
# -------------------------
rm = RiskManager(
    RiskManagerConfig(
        lambda_reg=0.5,
        lambda_q=0.5,
        warn_kfinal=1.10,
        veto_kfinal=1.35,
        min_confidence=0.15,
        size_curve=2.0,
        sl_curve=1.0,
        tp_curve=1.0,
        tp_reduce_in_high_risk=True,
    )
)

# -------------------------
# 4) Providers
# -------------------------
def risk_feature_provider(t, df):
    out = qra.run(df.iloc[: t + 1], horizon_bars=3)
    return {
        "k_final": out.q_factor,
        "regime_factor": out.regime_factor,
        "quantile_factor": out.q_factor,
        "level": out.level,
        "trend": out.trend,
    }

def intent_provider(t, df):
    intent = TradeIntent(
        direction="long",
        baseline_size=1.0,
        confidence=0.5,
        meta={"source": "mock"},
    )

    risk_out = qra.run(df.iloc[: t + 1], horizon_bars=3)
    rm_out = rm.decide(risk_out, intent)

    return {
        "direction": intent.direction,
        "baseline_size": intent.baseline_size * rm_out.action.scale_size,
        "confidence": intent.confidence,
        "accept_reject": rm_out.action.accept_reject,
        "scale_sl": rm_out.action.scale_sl,
        "scale_tp": rm_out.action.scale_tp,
        "k_final": rm_out.k_final,
    }

# -------------------------
# 5) Build Environment
# -------------------------
env = RiskEnv(
    df_ohlcv=df,
    cfg=RiskEnvConfig(
        initial_capital=10_000.0,
        max_position_units=1.0,
        base_sl_atr=1.5,
        base_tp_atr=2.0,
    ),
    risk_feature_provider=risk_feature_provider,
    intent_provider=intent_provider,
)

# -------------------------
# 6) Train PPO
# -------------------------
ppo_cfg = RiskPolicyConfig(
    algo="PPO",
    total_timesteps=20_000,
    eval_every_steps=4_000,
    eval_episodes=3,
    verbose=1,
)

ppo_policy = RiskPolicy(ppo_cfg)
ppo_policy.train(env)
ppo_policy.model.save("models/ppo_risk_20k")

print("✅ PPO training completed.")




Using cpu device
-----------------------------
| time/              |      |
|    fps             | 1    |
|    iterations      | 1    |
|    time_elapsed    | 1266 |
|    total_timesteps | 2048 |
-----------------------------
[Eval] mean=-0.0235
-----------------------------------------
| time/                   |             |
|    fps                  | 1           |
|    iterations           | 2           |
|    time_elapsed         | 2465        |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.006955652 |
|    clip_fraction        | 0.0266      |
|    clip_range           | 0.2         |
|    entropy_loss         | -5.67       |
|    explained_variance   | -3.92       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.0833      |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0175     |
|    std                  | 0.996       |
|    value_loss           | 0.411      

In [14]:
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from risk_env import RiskEnv, RiskEnvConfig
from risk_policy import RiskEnvGymWrapper
env = RiskEnv(
    df_ohlcv=df,                 # same dataframe
    cfg=RiskEnvConfig(
        initial_capital=10_000.0,
        max_position_units=1.0,
        base_sl_atr=1.5,
        base_tp_atr=2.0,
    ),
    risk_feature_provider=risk_feature_provider,
    intent_provider=intent_provider,
)
ppo_cfg = RiskPolicyConfig(
    algo="PPO",
    total_timesteps=20_000,
    eval_every_steps=4_000,
    eval_episodes=3,
    verbose=1,
)
env_vec = DummyVecEnv([
    lambda: RiskEnvGymWrapper(env, ppo_cfg)
])

env_vec = VecNormalize(
    env_vec,
    norm_obs=True,
    norm_reward=True,
)

# IMPORTANT: evaluation mode
env_vec.training = False
env_vec.norm_reward = False

model = PPO.load("models/ppo_risk_20k", env=env_vec)


obs = env_vec.reset()

total_reward = 0.0
total_pnl = 0.0

print("step | action | reward | pnl | capital | turnover")

for step in range(200):   # number of evaluation steps
    action, _ = model.predict(obs, deterministic=True)
    print("accept:", action[0][3], "size:", action[0][0])

    obs, reward, done, info = env_vec.step(action)
    

    # VecEnv info is list of dicts
    info = info[0]

    total_reward += reward[0]
    total_pnl += info.get("pnl", 0.0)

    print(
        f"{step:04d} | "
        f"{action[0]} | "
        f"{reward[0]:+.5f} | "
        f"{info.get('pnl', 0.0):+.4f} | "
        f"{info.get('capital', 0.0):.2f} | "
        f"{info.get('turnover', 0.0):.2f}"
    )

    if done:
        break
print("\n========== SUMMARY ==========")
print(f"Total reward : {total_reward:.4f}")
print(f"Total PnL    : {total_pnl:.2f}")
print(f"Final capital: {info.get('capital', 0.0):.2f}")


step | action | reward | pnl | capital | turnover
accept: 0.0 size: 0.0
0000 | [0.  0.5 0.5 0. ] | -0.00300 | +0.0000 | 10000.00 | 0.00
accept: 0.0 size: 0.0
0001 | [0.  0.5 0.5 0. ] | +0.00000 | +0.0000 | 10000.00 | 0.00
accept: 0.0 size: 0.0
0002 | [0.  0.5 0.5 0. ] | +0.00000 | +0.0000 | 10000.00 | 0.00
accept: 0.0 size: 0.0
0003 | [0.  0.5 0.5 0. ] | +0.00000 | +0.0000 | 10000.00 | 0.00
accept: 0.0 size: 0.0
0004 | [0.  0.5 0.5 0. ] | +0.00000 | +0.0000 | 10000.00 | 0.00
accept: 0.0 size: 0.0
0005 | [0.  0.5 0.5 0. ] | +0.00000 | +0.0000 | 10000.00 | 0.00
accept: 0.0 size: 0.0
0006 | [0.  0.5 0.5 0. ] | +0.00000 | +0.0000 | 10000.00 | 0.00
accept: 0.0 size: 0.0
0007 | [0.  0.5 0.5 0. ] | +0.00000 | +0.0000 | 10000.00 | 0.00
accept: 0.0 size: 0.0
0008 | [0.  0.5 0.5 0. ] | +0.00000 | +0.0000 | 10000.00 | 0.00
accept: 0.0 size: 0.0
0009 | [0.  0.5 0.5 0. ] | +0.00000 | +0.0000 | 10000.00 | 0.00
accept: 0.0 size: 0.0
0010 | [0.  0.5 0.5 0. ] | +0.00000 | +0.0000 | 10000.00 | 0.00
acce

In [7]:
from stable_baselines3 import PPO
from stable_baselines3 import SAC

ppo_model = PPO.load("models/ppo_risk_20k")

import numpy as np

def evaluate_agent(
    model,
    env,
    n_episodes=3,
    deterministic=True,
):
    import numpy as np

    results = {
        "episode_reward": [],
        "final_capital": [],
        "max_drawdown": [],
        "turnover": [],
    }

    for ep in range(n_episodes):

        # -------- reset (robust) --------
        reset_out = env.reset()
        if isinstance(reset_out, tuple):
            obs = reset_out[0]
        else:
            obs = reset_out

        done = False
        trunc = False

        total_reward = 0.0
        max_dd = 0.0
        total_turnover = 0.0

        while not (done or trunc):
            action, _ = model.predict(obs, deterministic=deterministic)

            step_out = env.step(action)

            # -------- step (robust) --------
            if len(step_out) == 5:
                obs, reward, done, trunc, info = step_out
            else:
                obs, reward, done, info = step_out
                trunc = False

            total_reward += float(reward)
            max_dd = max(max_dd, info.get("drawdown", 0.0))
            total_turnover += info.get("turnover", 0.0)

        results["episode_reward"].append(total_reward)
        results["final_capital"].append(info.get("capital", np.nan))
        results["max_drawdown"].append(max_dd)
        results["turnover"].append(total_turnover)

    return {
        "reward_mean": float(np.mean(results["episode_reward"])),
        "reward_std": float(np.std(results["episode_reward"])),
        "final_capital_mean": float(np.mean(results["final_capital"])),
        "max_drawdown_mean": float(np.mean(results["max_drawdown"])),
        "turnover_mean": float(np.mean(results["turnover"])),
    }



ppo_metrics = evaluate_agent(ppo_model, env, n_episodes=5)
print("PPO:", ppo_metrics)





PPO: {'reward_mean': -0.003, 'reward_std': 0.0, 'final_capital_mean': 10000.0, 'max_drawdown_mean': 0.0, 'turnover_mean': 0.0}


In [ ]:
# -------------------------
# 7) Train SAC
# -------------------------
sac_cfg = RiskPolicyConfig(
    algo="SAC",
    total_timesteps=20_000,
    eval_every_steps=4_000,
    eval_episodes=3,
    verbose=1,
)

sac_policy = RiskPolicy(sac_cfg)
sac_policy.train(env)
sac_policy.model.save("models/sac_risk_20k")

print("✅ SAC training completed.")

In [18]:
# ============================================================
#   FINAL TRAINING: RiskEnv + RiskPolicy (PPO) — OPTION B
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import os

from quantile_risk_agent import QuantileRiskAgent, RegimeConfig, QRConfig
from risk_manager import RiskManager, RiskManagerConfig, TradeIntent
from risk_env import RiskEnv, RiskEnvConfig
from risk_policy import RiskPolicy, RiskPolicyConfig

os.makedirs("models", exist_ok=True)

# ============================================================
# 1) Download OHLCV (DAILY)
# ============================================================
df = yf.download(
    tickers="AAPL",
    interval="1d",
    period="16mo",      # 15 mois train + 1 mois test
    auto_adjust=False,
    progress=False,
)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()

# Split train / test (last month = test)
train_df = df.iloc[:-22].copy()   # ~22 jours de trading = 1 mois
test_df  = df.iloc[-22:].copy()

print("Train bars:", len(train_df))
print("Test bars :", len(test_df))

# ============================================================
# 2) Quantile Risk Agent
# ============================================================
qra = QuantileRiskAgent(
    regime_config=RegimeConfig(
        quantiles=(0.25, 0.75),
        atr_window=14,
        trend_fraction=0.2,
        trend_window=20,
    ),
    qr_config=QRConfig(
        horizon_bars=3,
        feature_window=60,
        quantile_levels=(0.5, 0.95),
        alpha=0.01,
        alpha_mode="vol_scaled",
    ),
)

# ============================================================
# 3) Risk Manager (Option B)
# ============================================================
rm = RiskManager(
    RiskManagerConfig(
        lambda_reg=0.5,
        lambda_q=0.5,
        warn_kfinal=1.10,
        veto_kfinal=1.35,
        min_confidence=0.15,
        size_curve=2.0,
        sl_curve=1.0,
        tp_curve=1.0,
        tp_reduce_in_high_risk=True,
    )
)

# ============================================================
# 4) TradeIntent minimal mais réaliste (EMA + RSI)
# ============================================================
def intent_provider_minimal(t: int, df: pd.DataFrame):
    if t < 60:
        return {"direction": "flat", "baseline_size": 0.0, "confidence": 0.0}

    close = df["Close"].iloc[: t + 1]

    ema_fast = close.ewm(span=20, adjust=False).mean().iloc[-1]
    ema_slow = close.ewm(span=50, adjust=False).mean().iloc[-1]
    trend_strength = (ema_fast - ema_slow) / max(abs(ema_slow), 1e-9)

    delta = close.diff()
    up = delta.clip(lower=0).rolling(14).mean().iloc[-1]
    down = (-delta.clip(upper=0)).rolling(14).mean().iloc[-1]
    rs = up / max(down, 1e-9)
    rsi = 100 - (100 / (1 + rs))

    if abs(trend_strength) < 0.001:
        return {"direction": "flat", "baseline_size": 0.0, "confidence": 0.0}

    direction = "long" if trend_strength > 0 else "short"
    conf_trend = min(abs(trend_strength) / 0.01, 1.0)
    conf_rsi = min(abs(rsi - 50) / 25.0, 1.0)
    confidence = 0.6 * conf_trend + 0.4 * conf_rsi

    return {
        "direction": direction,
        "baseline_size": float(confidence),
        "confidence": float(confidence),
    }

# ============================================================
# 5) Providers (RiskManager + QuantileRiskAgent)
# ============================================================
def risk_feature_provider(t, df):
    out = qra.run(df.iloc[: t + 1], horizon_bars=3)
    return {
        "k_final": out.q_factor,
        "regime_factor": out.regime_factor,
        "quantile_factor": out.q_factor,
        "level": out.level,
        "trend": out.trend,
    }

def intent_provider(t, df):
    base = intent_provider_minimal(t, df)

    if base["direction"] == "flat":
        return {
            "direction": "flat",
            "baseline_size": 0.0,
            "confidence": 0.0,
            "accept_reject": 0,
            "scale_sl": 1.0,
            "scale_tp": 1.0,
        }

    intent = TradeIntent(
        direction=base["direction"],
        baseline_size=base["baseline_size"],
        confidence=base["confidence"],
        meta={"source": "EMA_RSI"},
    )

    risk_out = qra.run(df.iloc[: t + 1], horizon_bars=3)
    rm_out = rm.decide(risk_out, intent)

    return {
        "direction": intent.direction,
        "baseline_size": intent.baseline_size * rm_out.action.scale_size,
        "confidence": intent.confidence,
        "accept_reject": rm_out.action.accept_reject,
        "scale_sl": rm_out.action.scale_sl,
        "scale_tp": rm_out.action.scale_tp,
        "k_final": rm_out.k_final,
    }

# ============================================================
# 6) Build Training Environment
# ============================================================
env_train = RiskEnv(
    df_ohlcv=train_df,
    cfg=RiskEnvConfig(
        initial_capital=10_000.0,
        max_position_units=1.0,
        base_sl_atr=1.5,
        base_tp_atr=2.0,
    ),
    risk_feature_provider=risk_feature_provider,
    intent_provider=intent_provider,
)

# ============================================================
# 7) Train PPO (50k steps)
# ============================================================
ppo_cfg = RiskPolicyConfig(
    algo="PPO",
    total_timesteps=50_000,
    eval_every_steps=10_000,
    eval_episodes=3,
    verbose=1,
)

ppo_policy = RiskPolicy(ppo_cfg)
ppo_policy.train(env_train)
ppo_policy.model.save("models/ppo_risk_50k")

print("✅ PPO training completed (50k steps).")


Train bars: 312
Test bars : 22
Using cpu device
-----------------------------
| time/              |      |
|    fps             | 5    |
|    iterations      | 1    |
|    time_elapsed    | 358  |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 4           |
|    iterations           | 2           |
|    time_elapsed         | 824         |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.012203816 |
|    clip_fraction        | 0.0894      |
|    clip_range           | 0.2         |
|    entropy_loss         | -5.64       |
|    explained_variance   | -0.111      |
|    learning_rate        | 0.0003      |
|    loss                 | 0.0583      |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0175     |
|    std                  | 0.984       |
|    value_loss           | 

In [20]:
# ============================================================
#   FINAL EVALUATION SCRIPT — PPO Risk Agent (Option B)
# ============================================================

import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from risk_env import RiskEnv, RiskEnvConfig
from risk_policy import RiskEnvGymWrapper, RiskPolicyConfig

# ============================================================
# 1) Build evaluation dataframe (WARM-UP + TEST)
# ============================================================

df_eval = pd.concat([train_df, test_df], axis=0)
test_start_index = len(train_df)

print("Eval rows:", len(df_eval))
print("Test starts at index:", test_start_index)

# ============================================================
# 2) Build raw RiskEnv (needs enough history)
# ============================================================

env_eval_raw = RiskEnv(
    df_ohlcv=df_eval,
    cfg=RiskEnvConfig(
        initial_capital=10_000.0,
        max_position_units=1.0,
        base_sl_atr=1.5,
        base_tp_atr=2.0,
    ),
    risk_feature_provider=risk_feature_provider,
    intent_provider=intent_provider,
)

# Dummy cfg just for wrapper bounds
ppo_cfg = RiskPolicyConfig(algo="PPO", total_timesteps=1)

# Wrap exactly like training
env_eval = DummyVecEnv([
    lambda: RiskEnvGymWrapper(env_eval_raw, ppo_cfg)
])

# If VecNormalize was NOT saved during training
env_eval = VecNormalize(env_eval, norm_obs=True, norm_reward=False)
env_eval.training = False
env_eval.norm_reward = False

# ============================================================
# 3) Load trained model
# ============================================================

model = PPO.load("models/ppo_risk_50k", env=env_eval)

# ============================================================
# 4) Evaluation function (metrics ONLY on test window)
# ============================================================

def evaluate_sb3_vecenv(
    model,
    vec_env,
    test_start_index,
    n_episodes=5,
    deterministic=True,
    max_steps=10_000,
):
    results = {
        "episode_reward": [],
        "final_capital": [],
        "max_drawdown": [],
        "turnover": [],
        "total_pnl": [],
        "trade_steps": [],
        "steps": [],
    }

    for ep in range(n_episodes):
        obs = vec_env.reset()

        ep_reward = 0.0
        ep_turnover = 0.0
        ep_pnl = 0.0
        ep_max_dd = 0.0
        ep_trade_steps = 0
        ep_steps = 0

        for _ in range(max_steps):
            action, _ = model.predict(obs, deterministic=deterministic)

            obs, reward, done, infos = vec_env.step(action)
            info = infos[0]

            current_t = vec_env.envs[0].env.t  # current timestep in RiskEnv

            # -------- COUNT METRICS ONLY IN TEST WINDOW --------
            if current_t >= test_start_index:
                ep_reward += float(reward[0])
                ep_pnl += float(info.get("pnl", 0.0))
                ep_turnover += float(info.get("turnover", 0.0))
                ep_max_dd = max(ep_max_dd, float(info.get("drawdown", 0.0)))

                if float(info.get("turnover", 0.0)) > 1e-9:
                    ep_trade_steps += 1

                ep_steps += 1

            if bool(done[0]):
                final_capital = float(info.get("capital", np.nan))
                break
        else:
            final_capital = float(info.get("capital", np.nan))

        results["episode_reward"].append(ep_reward)
        results["final_capital"].append(final_capital)
        results["max_drawdown"].append(ep_max_dd)
        results["turnover"].append(ep_turnover)
        results["total_pnl"].append(ep_pnl)
        results["trade_steps"].append(ep_trade_steps)
        results["steps"].append(ep_steps)

    return {
        "reward_mean": float(np.mean(results["episode_reward"])),
        "reward_std": float(np.std(results["episode_reward"])),
        "final_capital_mean": float(np.mean(results["final_capital"])),
        "final_capital_std": float(np.std(results["final_capital"])),
        "total_pnl_mean": float(np.mean(results["total_pnl"])),
        "max_drawdown_mean": float(np.mean(results["max_drawdown"])),
        "turnover_mean": float(np.mean(results["turnover"])),
        "trade_steps_mean": float(np.mean(results["trade_steps"])),
        "test_steps_mean": float(np.mean(results["steps"])),
    }

# ============================================================
# 5) Run evaluation
# ============================================================

metrics = evaluate_sb3_vecenv(
    model,
    env_eval,
    test_start_index=test_start_index,
    n_episodes=10,
    deterministic=True,
)

print("\n========== PPO EVALUATION (LAST MONTH) ==========")
for k, v in metrics.items():
    print(f"{k:20s}: {v:.6f}")


Eval rows: 334
Test starts at index: 312

========== PPO EVALUATION (LAST MONTH) ==========
reward_mean         : 0.000000
reward_std          : 0.000000
final_capital_mean  : 10000.000000
final_capital_std   : 0.000000
total_pnl_mean      : 0.000000
max_drawdown_mean   : 0.000000
turnover_mean       : 0.000000
trade_steps_mean    : 0.000000
test_steps_mean     : 21.000000
